In [1]:
pip install lightgbm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Mohamed Adel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd # Data manipulation
from sklearn.model_selection import train_test_split # Splitting the data
from sklearn.metrics import mean_squared_error, r2_score # Evaluating the model
from sklearn.ensemble import RandomForestRegressor # Model
from sklearn.svm import SVR # Model
from sklearn.neighbors import KNeighborsRegressor # Model
from sklearn.ensemble import GradientBoostingRegressor #Model
import lightgbm as lgb

# Load your dataset
df = pd.read_excel(r'C:\Users\Mohamed Adel\Documents\KH-17 Excell result.xlsx')
y = pd.read_excel(r'C:\Users\Mohamed Adel\Documents\KH-17 Excell result.xlsx')


df = df.iloc[:4954, :5] # rows, columns

In [14]:
y = y['DRAD'].iloc[:4954] # target

In [15]:
y

0       0.026054
1       0.035024
2       0.029268
3       0.010824
4      -0.029543
          ...   
4949   -0.129322
4950   -0.032787
4951    0.419461
4952    0.476131
4953    0.601183
Name: DRAD, Length: 4954, dtype: float64

In [ ]:
# # Calculations on averages
# avg_uran = df['eU'].mean() # check for use 'eU' or 'URAN' based on excel file
# avg_thor = df['THOR'].mean()
# avg_k = df['K'].mean()

# df['eUi'] = (avg_uran / avg_thor) * df['THOR']
# df['ki'] = (avg_k / avg_thor) * df['THOR']

# # Calculattion for percentages
# df['eUD%'] = (df['eU'] - df['eUi']) / df['eU'] # check for use 'eU' or 'URAN' based on excel file
# df['KD%'] = (df['K'] - df['ki']) / df['K']

# # Calculate DRAD
# df['DRAD'] = df['eUD%'] - df['KD%']

In [16]:
df

,MD,GR,NPHI,RHOB,RT
0,3945.2998,42.0540,0.2718,2.2235,1.1550
1,3945.7998,44.6129,0.3469,2.1352,1.2407
2,3946.2998,44.9639,0.3767,2.2745,1.3228
3,3946.7998,43.3759,0.3300,2.4704,1.2631
4,3947.2998,43.5552,0.2448,2.5271,1.3076
...,...,...,...,...,...
4949,6419.7998,33.6318,0.1022,2.6075,2.4332
4950,6420.2998,35.7521,0.1322,2.5986,1.9425
4951,6420.7998,41.6645,0.1727,2.5944,1.5841
4952,6421.2998,47.8876,0.2240,2.5366,1.3535


In [17]:
df.columns

Index(['MD', 'GR', 'NPHI', 'RHOB', 'RT'], dtype='object')

In [18]:
from sklearn.linear_model import LinearRegression

# Define features and target
X = df.values # features / inputs
y = y.values # target

# Splitting
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 20% to 30%

# LR Model
model = LinearRegression()

# Training
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Evaluating
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Squared Error: {mse}')
print(f'R-squared: {r2}')


Mean Squared Error: 0.530455131173829
R-squared: 0.19502208257755038


In [19]:
df['Predicted_DARD'] = model.predict(X)

# Export the DataFrame to excel file
df.to_excel('updated_dataset_with_DRAD_KH-17.xlsx', index=False)

# Show the Equation
intercept = model.intercept_
coefficients = model.coef_
feature_names = df.columns

print(f'Linear Regression Equation:')
print(f'y = {intercept:.4f}', end='')
for coef, feature in zip(coefficients, feature_names):
    print(f' + {coef:.4f} * {feature}', end='')
print()


Linear Regression Equation:
y = 1.3109 + -0.0001 * MD + 0.0327 * GR + -3.4968 * NPHI + -0.5138 * RHOB + 0.0001 * RT


In [20]:
# Creating a function to train and evaluate the models
def train_and_evaluate(model, X_train, X_test, y_train, y_test):
    # Training
    model.fit(X_train, y_train)

    # Prediction
    y_pred = model.predict(X_test)

    # Evaluating
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    return mse, r2, y_pred


In [21]:
# Defining the models
models = {
    'Linear Regression': LinearRegression(), # key: value
    'Random Forest': RandomForestRegressor(),
    'Support Vector Regressor': SVR(),
    'K-Nearest Neighbors': KNeighborsRegressor(),
    'Gradient Boosting': GradientBoostingRegressor(),
    'LightGBM': lgb.LGBMRegressor()
}

# Creating dictionary to save the models results (MSE, R-squared)
results = {}

for name, model in models.items():
    mse, r2, y_pred = train_and_evaluate(model, X_train, X_test, y_train, y_test)
    results[name] = {'MSE': mse, 'R-squared': r2}

    # Saving prediction for each model in a new column
    df[f'Predicted_{name.replace(" ", "_")}'] = model.predict(X) # predicted DRAD column as a new column in the dataframe

# Exportig the new genrated target to an excel file
df.to_excel(r'C:\Users\Mohamed Adel\Desktop\updated_dataset_with_predictions_KH-17.xlsx', index=False)

# Printing the MSE and R-squared for each model
for name, metrics in results.items():
    print(f'{name}:')
    print(f'  Mean Squared Error: {metrics["MSE"]}')
    print(f'  R-squared: {metrics["R-squared"]}')
    print()

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000877 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 3963, number of used features: 5
[LightGBM] [Info] Start training from score -0.271142


C:\Users\Mohamed Adel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\Mohamed Adel\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Linear Regression:
  Mean Squared Error: 0.530455131173829
  R-squared: 0.19502208257755038

Random Forest:
  Mean Squared Error: 0.17715594165427476
  R-squared: 0.7311617654516773

Support Vector Regressor:
  Mean Squared Error: 0.6520913328763568
  R-squared: 0.010436336158233583

K-Nearest Neighbors:
  Mean Squared Error: 0.13189255787624485
  R-squared: 0.7998499961197517

Gradient Boosting:
  Mean Squared Error: 0.33567930910056426
  R-squared: 0.4905988927590743

LightGBM:
  Mean Squared Error: 0.2734217070878083
  R-squared: 0.5850762422401582

